# A-Share Multi-Factor Quantitative Research Platform
## Complete Research Workflow

This notebook demonstrates the full research workflow:
1. Data loading and exploration
2. Factor computation and evaluation
3. Alpha signal construction
4. Portfolio optimization comparison
5. Backtest and performance analysis
6. Risk decomposition

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({'figure.facecolor': 'white', 'axes.grid': True, 'grid.alpha': 0.3})
%matplotlib inline

## 1. Data Loading

Load synthetic A-share data with embedded alpha structure.

In [ ]:
from quant_platform.data.pipeline import DataPipeline
from quant_platform.data.providers.synthetic import SyntheticDataProvider

provider = SyntheticDataProvider(n_stocks=200, start_date="2022-01-01", end_date="2024-12-31")
pipeline = DataPipeline(provider=provider, start_date="2022-01-01", end_date="2024-12-31")
pipeline.run()

prices = pipeline.get_close()
returns = pipeline.returns
benchmark = pipeline.benchmark
metadata = pipeline.metadata
financials = pipeline.financials.unstack("asset")

print(f"Assets: {prices.shape[1]}, Dates: {prices.shape[0]}")
print(f"Sectors: {metadata['sector'].nunique()}")
print(f"ST stocks: {metadata['is_st'].sum()}")

In [ ]:
# Quick data inspection
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Sector distribution
metadata['sector'].value_counts().head(10).plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 Sectors by Stock Count')
axes[0].set_ylabel('Number of Stocks')

# Cumulative benchmark vs equal-weight
cum_bench = (1 + benchmark).cumprod()
cum_ew = (1 + returns.mean(axis=1)).cumprod()
axes[1].plot(cum_bench.index, cum_bench.values, label='Benchmark', linewidth=1.5)
axes[1].plot(cum_ew.index, cum_ew.values, label='Equal-Weight', linewidth=1.5, alpha=0.7)
axes[1].set_title('Cumulative Returns')
axes[1].legend()
plt.tight_layout()
plt.show()

## 2. Factor Computation & Evaluation

Compute all 15 factors, process, and evaluate with Rank IC.

In [ ]:
from quant_platform.factors.evaluation import factor_correlation, ic_summary, rank_ic
from quant_platform.factors.fundamental import register_all as reg_fund
from quant_platform.factors.processing import process_factor
from quant_platform.factors.registry import get_registry
from quant_platform.factors.technical import register_all as reg_tech

reg_tech()
reg_fund()
registry = get_registry()

# Compute raw factors
raw_factors = {}
for fn in registry.list_all():
    cls = registry.get(fn)
    inst = cls()
    try:
        if inst.category.value == 'fundamental' and financials is not None:
            result = inst.run(prices, financials)
        else:
            result = inst.run(prices)
        raw_factors[result.name] = result.values
    except Exception as e:
        print(f"  Failed: {fn} - {e}")

print(f"Computed {len(raw_factors)} factors")

In [ ]:
# Process and evaluate
sector_map = metadata['sector']
mcap = financials['market_cap'] if 'market_cap' in financials.columns else None

processed = {}
ic_results = {}
for name, factor in raw_factors.items():
    processed[name] = process_factor(factor, sector_map=sector_map, market_cap=mcap)
    ic = rank_ic(processed[name], returns)
    ic_results[name] = ic_summary(ic)

# IC bar chart
ic_df = pd.DataFrame(ic_results).T.sort_values('mean_ic', ascending=True)
colors = ['#d62728' if x < 0 else '#2ca02c' for x in ic_df['mean_ic']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(ic_df.index, ic_df['mean_ic'], color=colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Mean Rank IC')
ax.set_title('Factor Information Coefficient (Rank IC)')
plt.tight_layout()
plt.show()

In [ ]:
# Factor correlation matrix
corr = factor_correlation(processed)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, ax=ax,
            cbar_kws={'shrink': 0.7})
ax.set_title('Factor Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Alpha Signal Construction

Combine factors using ICIR-weighted method.

In [ ]:
from quant_platform.alpha.pipeline import AlphaPipeline

alpha = AlphaPipeline(method='icir_weighted', lookback=252, min_icir=0.0)
signal = alpha.run(processed, returns)

print(f"Signal shape: {signal.shape}")
print(f"Signal range: [{signal.min().min():.3f}, {signal.max().max():.3f}]")

## 4. Portfolio Optimization — Compare All Methods

Run backtests with all three optimizers and compare.

In [ ]:
from quant_platform.backtest.cost_model import CostModel
from quant_platform.backtest.engine import BacktestEngine
from quant_platform.portfolio.constraints import PortfolioConstraints

constraints = PortfolioConstraints(
    long_only=True, max_weight=0.05, max_sector_exposure=0.30,
    max_turnover=0.30, lot_size=100,
)
cost_model = CostModel(commission=0.0003, stamp_tax=0.001, slippage=0.001)

all_results = {}
for opt in ['equal_weight', 'mean_variance', 'risk_parity']:
    print(f"Running {opt}...")
    engine = BacktestEngine(
        initial_capital=10_000_000,
        rebalance_frequency='monthly',
        cost_model=cost_model,
        constraints=constraints,
        optimizer=opt,
        covariance_method='ledoit_wolf',
        covariance_lookback=252,
    )
    all_results[opt] = engine.run(
        signal=signal, prices=prices, returns=returns,
        benchmark_returns=benchmark, sector_map=sector_map,
        financials=financials,
    )

In [ ]:
# Comparison table
comparison = []
for opt, res in all_results.items():
    s = res['summary']
    comparison.append({
        'Optimizer': opt,
        'Total Return': f"{s['total_return']*100:.1f}%",
        'Ann. Return': f"{s['annual_return']*100:.1f}%",
        'Sharpe': round(s['sharpe_ratio'], 2),
        'Sortino': round(s['sortino_ratio'], 2),
        'Max DD': f"{s['max_drawdown']*100:.1f}%",
        'Calmar': round(s['calmar_ratio'], 2),
        'Win Rate': f"{s['win_rate']*100:.1f}%",
    })
display(pd.DataFrame(comparison))

In [ ]:
# Equity curve comparison
fig, ax = plt.subplots(figsize=(14, 6))

colors = {'equal_weight': '#1f77b4', 'mean_variance': '#ff7f0e', 'risk_parity': '#2ca02c'}
for opt, res in all_results.items():
    curve = (1 + res['daily_returns']).cumprod()
    ax.plot(curve.index, curve.values, label=opt, color=colors[opt], linewidth=1.5)

# Benchmark
bench_curve = (1 + benchmark.reindex(curve.index, fill_value=0)).cumprod()
ax.plot(bench_curve.index, bench_curve.values, 'k--', label='Benchmark', linewidth=1, alpha=0.5)

ax.set_title('Strategy Comparison — Cumulative Returns', fontsize=14, fontweight='bold')
ax.set_ylabel('Cumulative Return (x)')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}x'))
plt.tight_layout()
plt.show()

## 5. Risk Analysis

VaR, CVaR, and stress tests for the best strategy.

In [ ]:
from quant_platform.risk.stress import run_all_stress_tests
from quant_platform.risk.var import var_summary

# Use equal_weight for analysis (most robust)
strat_returns = all_results['equal_weight']['daily_returns']

risk = var_summary(strat_returns)
print("Risk Metrics:")
for k, v in risk.items():
    if isinstance(v, float):
        print(f"  {k}: {v*100:.2f}%" if 'var' in k or 'cvar' in k else f"  {k}: {v:.2f}")

stress = run_all_stress_tests(strat_returns)
print("\nStress Tests:")
display(stress)

In [ ]:
# Drawdown and Rolling Sharpe
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Drawdown
cumulative = (1 + strat_returns).cumprod()
running_max = cumulative.expanding(min_periods=1).max()
drawdown = (cumulative / running_max - 1) * 100
ax1.fill_between(drawdown.index, 0, drawdown.values, color='#d62728', alpha=0.3)
ax1.plot(drawdown.index, drawdown.values, color='#d62728', linewidth=0.5)
ax1.set_title('Drawdown (% from Peak)', fontweight='bold')
ax1.set_ylabel('Drawdown (%)')

# Rolling Sharpe
window = 126
roll_ret = strat_returns.rolling(window).mean() * 252
roll_vol = strat_returns.rolling(window).std() * np.sqrt(252)
roll_sharpe = roll_ret / roll_vol
ax2.plot(roll_sharpe.index, roll_sharpe.values, color='#2ca02c', linewidth=1)
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax2.set_title(f'Rolling {window}-Day Sharpe Ratio', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. LLM Sentiment Factor (Bonus)

Demonstrate the LLM-augmented sentiment factor.

In [ ]:
from quant_platform.agent.sentiment_factor import LLMSentimentFactor

sent_factor = LLMSentimentFactor(use_real_llm=False, lookback_days=5)
sentiment = sent_factor.compute(prices)

fig, ax = plt.subplots(figsize=(14, 4))
avg_sentiment = sentiment.mean(axis=1)
ax.plot(avg_sentiment.index, avg_sentiment.values, color='#9467bd', linewidth=1)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Average LLM Sentiment Score Over Time', fontweight='bold')
ax.set_ylabel('Sentiment (-1 to +1)')
plt.tight_layout()
plt.show()

print(f"Mean sentiment: {sentiment.mean().mean():.3f}")
print(f"Std sentiment: {sentiment.std().mean():.3f}")

## Summary

This notebook demonstrated:
- Data pipeline with 200 A-share stocks, 3 years
- 15 factor computation, cross-sectional processing, IC evaluation
- ICIR-weighted alpha signal construction
- Comparison of equal-weight, mean-variance, and risk parity optimizers
- Risk analysis: VaR, drawdown, rolling Sharpe, stress tests
- LLM sentiment factor integration

The platform is production-ready with:
- 105 unit tests, 68%+ code coverage
- Numba JIT acceleration (5-20x speedup)
- Real A-share data support via Tushare
- 10 market-specific edge cases handled
- Config-driven, ABC-based architecture
- CLI for run/analyze/compare/sweep/cache